# 1.4 Assign children aged 0–9 to HDB blocks by flat type

No dataset gives the number of children per HDB block, so demand has to be disaggregated from the
subzone level. SingStat's "Resident Population by Planning Area/Subzone, Age Group and Type of
Dwelling" (June 2026) says, for each subzone, how many children aged 0–4 and 5–9 live in 1–2 room,
3-room, 4-room and 5-room/executive HDB flats, and HDB Property Information (1_02) counts each
block's flats with the same split. Children in a subzone's 4-room flats are therefore spread over
that subzone's 4-room flats in proportion to each block's count, and likewise for the other types:

    child_pop(block, age, type) = subzone_children(age, type) × block_units(type) / subzone_hdb_units(type)

This replaces the earlier floor-area method (Kenneth, see
`../0_Data_Exploration/assign_child_population_floor_area.ipynb`), decided 2026-10-22: it needs no
resale download or floor-area estimate, and every subzone's children are placed exactly. Fallback:
if a subzone has children in a flat type but none of its matched blocks has that type, those
children are spread by total dwelling units instead (`used_fallback`). Original script:
`scripts/assign_child_population_fillnull.py` (Jessie).

**Inputs**
- `output/hdb_blocks.gpkg`, layer `hdb_blocks` (1_02) — residential blocks with unit counts by type.
- `data/MasterPlanSubZones_2019.geojson` — subzone polygons (names match SingStat's).
- `data/subzone_child_population_2026.xlsx` — SingStat `respopagesextod2026e.xlsx`, sheet `2026(Total)`.

**Outputs**
- `output/hdb_blocks_children_final.gpkg`, layer `hdb_blocks_children` — 10,769 residential blocks
  with `child_0_4`, `child_5_9`, `child_pop_0_9`, the four flat-type unit columns and `used_fallback`;
  read by 1_08 and the ESDA / hypothesis notebooks.
- `output/child_population_unassigned.csv` — subzone × age × flat-type cells whose children could
  not be placed on any block (subzones with HDB children but no matched residential block).

In [1]:
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "bt4015").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from bt4015.paths import DATA, OUT, FIG, LARGE

HDB_GPKG = OUT / "hdb_blocks.gpkg"
SUBZONE_GEOJSON = DATA / "MasterPlanSubZones_2019.geojson"
POP_XLSX = DATA / "subzone_child_population_2026.xlsx"
POP_SHEET = "2026(Total)"

OUT_GPKG = OUT / "hdb_blocks_children_final.gpkg"
OUT_LAYER = "hdb_blocks_children"
UNASSIGNED_CSV = OUT / "child_population_unassigned.csv"

AGE_GROUPS = {"0 - 4": "child_0_4", "5 - 9": "child_5_9"}   # SingStat label -> output column

# SingStat flat type -> HDB Property Information unit columns.
# "1- and 2-Room Flats" includes studio apartments (SingStat footnote *).
FLAT_TYPES = {
    "1- and 2-Room Flats*": ["1room_sold", "2room_sold", "studio_apartment_sold",
                             "1room_rental", "2room_rental", "other_room_rental"],
    "3-Room Flats": ["3room_sold", "3room_rental"],
    "4-Room Flats": ["4room_sold"],
    "5-Room and Executive Flats": ["5room_sold", "exec_sold", "multigen_sold"],
}

## 1. Residential blocks with units by flat type

The four SingStat flat-type groups are built from the HDB unit columns, so each block's units
line up with the population table's categories.

In [2]:
# 1. Residential blocks only (carparks, markets etc. carry no children); block_id = row position.
hdb = gpd.read_file(HDB_GPKG, layer="hdb_blocks")
hdb = hdb[hdb["is_residential"]].reset_index(drop=True)
hdb["block_id"] = hdb.index

# 2. Units per SingStat flat-type group; null unit columns count as zero.
for flat_type, cols in FLAT_TYPES.items():
    hdb[flat_type] = hdb[cols].fillna(0).sum(axis=1)
print(f"Residential HDB blocks: {len(hdb):,}")
print("Units by flat type:")
print(hdb[list(FLAT_TYPES)].sum().astype(int).to_string())

Residential HDB blocks: 10,769
Units by flat type:
1- and 2-Room Flats*          122756
3-Room Flats                  253898
4-Room Flats                  466809
5-Room and Executive Flats    328602


## 2. Tag each block with its subzone

Blocks are located by centroid; SingStat names subzones the same way as the 2019 Master Plan
(upper-cased), so the name is the join key.

In [3]:
# 1. Subzone polygons in the blocks' CRS; one row per block even if a centroid touched two polygons.
subzones = gpd.read_file(SUBZONE_GEOJSON)[["SUBZONE_N", "PLN_AREA_N", "geometry"]].to_crs(hdb.crs)
centroids = gpd.GeoDataFrame(hdb[["block_id"]], geometry=hdb.centroid, crs=hdb.crs)
located = gpd.sjoin(centroids, subzones, how="left", predicate="within").drop_duplicates("block_id")
hdb = hdb.merge(located[["block_id", "SUBZONE_N", "PLN_AREA_N"]], on="block_id", how="left")
hdb = hdb.rename(columns={"SUBZONE_N": "subzone", "PLN_AREA_N": "planning_area"})
print(f"Blocks with no subzone (centroid outside every polygon): {int(hdb['subzone'].isna().sum())}")
print(f"Subzones with residential HDB: {hdb['subzone'].nunique()}")

Blocks with no subzone (centroid outside every polygon): 0
Subzones with residential HDB: 159


## 3. SingStat children by subzone, age group and flat type

The sheet has a three-row header; "-" means nil. Only the two child age groups and the four HDB
flat types are kept (private housing and "Total" rows are dropped), which is why the national
total here is HDB children only.

In [4]:
def load_child_pop(path):
    """Children by subzone x age group x flat type, long format."""
    df = pd.read_excel(path, sheet_name=POP_SHEET, header=2)
    df.columns = ["planning_area", "subzone", "age", "flat_type", "pop"]
    df = df[df["age"].isin(AGE_GROUPS) & df["flat_type"].isin(FLAT_TYPES)
            & df["subzone"].ne("Total")].copy()
    df["pop"] = pd.to_numeric(df["pop"].replace("-", 0))  # "-" means nil
    df["subzone"] = df["subzone"].str.strip().str.upper()
    return df[["subzone", "age", "flat_type", "pop"]]


# 1. Load and check that every SingStat subzone name exists in the map.
pop = load_child_pop(POP_XLSX)
missing_names = set(pop["subzone"]) - set(subzones["SUBZONE_N"])
if missing_names:
    print(f"WARNING: {len(missing_names)} population subzones not in the subzone map: {sorted(missing_names)}")
print(f"Population cells: {len(pop):,} (subzone x age x flat type); subzones: {pop['subzone'].nunique()}")
print("HDB children by age group and flat type:")
print(pop.pivot_table(index="flat_type", columns="age", values="pop", aggfunc="sum", margins=True).astype(int).to_string())

Population cells: 2,656 (subzone x age x flat type); subzones: 332
HDB children by age group and flat type:
age                          0 - 4   5 - 9     All
flat_type                                         
1- and 2-Room Flats*          8430    7860   16290
3-Room Flats                 13130   14360   27490
4-Room Flats                 59310   61490  120800
5-Room and Executive Flats   41770   53560   95330
All                         122640  137270  259910


## 4. Dasymetric split, one row per block × age × flat type

Working in long format means every share is computed by key (subzone, age, flat type), never by
row order. `share` is the block's units of that type over the subzone's; when the subzone has no
block of that type but SingStat reports children in it, the share falls back to total dwelling
units (`used_fallback`).

In [5]:
# 1. Block units in long format, joined to the population cells of the same subzone and flat type.
units = hdb.melt(id_vars=["block_id", "subzone", "total_dwelling_units"],
                 value_vars=list(FLAT_TYPES), var_name="flat_type", value_name="units")
cells = units.merge(pop, on=["subzone", "flat_type"], how="inner")

# 2. Denominators per (subzone, age, flat type): units of that type, and all units for the fallback.
cells["type_total"] = cells.groupby(["subzone", "age", "flat_type"])["units"].transform("sum")
cells["all_total"] = cells.groupby(["subzone", "age", "flat_type"])["total_dwelling_units"].transform("sum")
cells["used_fallback"] = cells["type_total"].eq(0) & cells["pop"].gt(0)

# 3. Share and children; a cell with no units at all gives NaN -> 0.
cells["share"] = (cells["units"] / cells["type_total"]).where(
    ~cells["used_fallback"], cells["total_dwelling_units"] / cells["all_total"]
).fillna(0)
cells["children"] = cells["pop"] * cells["share"]
print(f"Cells: {len(cells):,}; fallback cells: {int(cells['used_fallback'].sum()):,}")

Cells: 86,152; fallback cells: 0


## 5. Children per block

In [6]:
# 1. Sum over flat types to one column per age group, then the 0-9 total.
per_block = cells.pivot_table(index="block_id", columns="age", values="children", aggfunc="sum")
per_block = per_block.rename(columns=AGE_GROUPS).reindex(columns=list(AGE_GROUPS.values()))
hdb = hdb.merge(per_block, left_on="block_id", right_index=True, how="left")
hdb["child_pop_0_9"] = hdb["child_0_4"] + hdb["child_5_9"]

# 2. Flag blocks that received children through the fallback rule.
fallback_blocks = set(cells.loc[cells["used_fallback"] & cells["children"].gt(0), "block_id"])
hdb["used_fallback"] = hdb["block_id"].isin(fallback_blocks)
print(f"Blocks with children: {int(hdb['child_pop_0_9'].gt(0).sum()):,}; "
      f"median per block {hdb['child_pop_0_9'].median():.1f}, max {hdb['child_pop_0_9'].max():.0f}")

Blocks with children: 10,755; median per block 20.6, max 250


## 6. Children that reached no block

A cell is unassigned when its subzone has HDB children in SingStat but no matched residential block
(e.g. blocks still under the HDB listing but absent from the polygons). They are listed, not
silently dropped.

In [7]:
# 1. Placed vs published per cell; the difference is what could not be assigned.
placed = cells.groupby(["subzone", "age", "flat_type"])["children"].sum()
totals = pop.set_index(["subzone", "age", "flat_type"])["pop"]
lost = (totals - placed.reindex(totals.index).fillna(0)).round(6)
lost = lost[lost > 0].rename("unassigned_children").reset_index()
lost.to_csv(UNASSIGNED_CSV, index=False)

total = pop["pop"].sum()
assigned = hdb["child_pop_0_9"].sum()
fb_children = cells.loc[cells["used_fallback"], "children"].sum()
print(f"HDB children aged 0-9 (sum of flat types): {total:,.0f}")
print(f"  assigned to blocks : {assigned:,.0f} ({assigned / total:.2%})")
print(f"    via fallback     : {fb_children:,.0f} children, {len(fallback_blocks)} blocks")
print(f"  unassigned         : {lost['unassigned_children'].sum():,.0f} "
      f"in {lost['subzone'].nunique()} subzones -> {UNASSIGNED_CSV.name}")
print(f"Blocks with null child_pop_0_9: {int(hdb['child_pop_0_9'].isna().sum())}")
print(f"Blocks with zero child_pop_0_9: {int(hdb['child_pop_0_9'].eq(0).sum())}")
if len(lost):
    print(lost.to_string(index=False))

HDB children aged 0-9 (sum of flat types): 259,910
  assigned to blocks : 259,910 (100.00%)
    via fallback     : 0 children, 0 blocks
  unassigned         : 0 in 0 subzones -> child_population_unassigned.csv
Blocks with null child_pop_0_9: 0
Blocks with zero child_pop_0_9: 14


## 7. Write the layer

In [8]:
keep = ["join_key", "BLK_NO", "POSTAL_COD", "subzone", "planning_area", "match_status",
        "review_flag", "total_dwelling_units", "gfa_m2", *FLAT_TYPES, "child_0_4", "child_5_9",
        "child_pop_0_9", "used_fallback", "geometry"]
hdb[keep].to_file(OUT_GPKG, layer=OUT_LAYER, driver="GPKG")
print(f"Wrote layer '{OUT_LAYER}' ({len(hdb):,} blocks) to {OUT_GPKG.relative_to(ROOT)}: "
      f"children 0-4 {hdb['child_0_4'].sum():,.0f}, 5-9 {hdb['child_5_9'].sum():,.0f}, 0-9 {hdb['child_pop_0_9'].sum():,.0f}")
print(f"Wrote {UNASSIGNED_CSV.relative_to(ROOT)} ({len(lost)} rows)")

Wrote layer 'hdb_blocks_children' (10,769 blocks) to output/hdb_blocks_children_final.gpkg: children 0-4 122,640, 5-9 137,270, 0-9 259,910
Wrote output/child_population_unassigned.csv (0 rows)
